### Regresion

In [1]:
"""
XGBoost para Regresión y Monitoreo de Convergencia.
Dataset Real: Diabetes Dataset (Efron et al., 2004 / Scikit-Learn).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
from sklearn.inspection import permutation_importance
import xgboost as xgb

# ==============================================================================
# 1. Ingesta de Datos y Partición de Prueba Independiente
# ==============================================================================
diab = load_diabetes()
X = pd.DataFrame(diab.data, columns=diab.feature_names)

# Renombrado de variables para interpretación clínica directa
nombres_clinicos = {
    'age': 'Edad', 'sex': 'Sexo', 'bmi': 'IMC (BMI)', 'bp': 'Presión Art.',
    's1': 'Colesterol', 's2': 'LDL', 's3': 'HDL', 's4': 'TCH',
    's5': 'Triglicéridos (LTG)', 's6': 'Glucosa'
}
X = X.rename(columns=nombres_clinicos)
y = diab.target

# Partición 75% Entrenamiento y 25% Prueba independiente
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

# ==============================================================================
# 2. Configuración y Entrenamiento de XGBRegressor con Monitoreo
# ==============================================================================
modelo_xgb = xgb.XGBRegressor(
    n_estimators=120,        # Número de árboles secuenciales
    learning_rate=0.04,      # Factor de contracción (shrinkage eta)
    max_depth=3,             # Árboles poco profundos para regularizar sesgo/varianza
    reg_lambda=2.0,          # Regularización L2 sobre los pesos de las hojas
    reg_alpha=0.3,           # Regularización L1 (sparsity)
    subsample=0.8,           # 80% de observaciones por árbol
    colsample_bytree=0.8,    # 80% de variables evaluadas por árbol
    eval_metric='rmse',      # Métrica de monitoreo
    random_state=42,
    n_jobs=-1
)

# Monitoreo simultáneo en entrenamiento y prueba para auditar sobreajuste
eval_set = [(X_tr, y_tr), (X_te, y_te)]
modelo_xgb.fit(X_tr, y_tr, eval_set=eval_set, verbose=False)

# ==============================================================================
# 3. Inferencia y Evaluación en el Conjunto de Test
# ==============================================================================
y_pred = modelo_xgb.predict(X_te)
r2_val = r2_score(y_te, y_pred)
rmse_val = root_mean_squared_error(y_te, y_pred)
mae_val = mean_absolute_error(y_te, y_pred)

print("=" * 65)
print("EVALUACIÓN DE REGRESIÓN: XGBOOST REGRESSOR (TEST SET)")
print("=" * 65)
print(f"Coeficiente de Determinación (R²) : {r2_val:.4f} ({r2_val*100:.1f}% varianza explicada)")
print(f"Error Cuadrático Medio (RMSE)     : {rmse_val:.2f} puntos")
print(f"Error Absoluto Medio (MAE)        : {mae_val:.2f} puntos")

# ==============================================================================
# 4. Cálculo de Importancia de Atributos: Ganancia (Gain) vs. Permutación
# ==============================================================================
importancias_gain = modelo_xgb.feature_importances_
perm_res = permutation_importance(modelo_xgb, X_te, y_te, n_repeats=10, random_state=42, scoring='r2')

top_idx = np.argsort(importancias_gain)
vars_ordenadas = [X.columns[i] for i in top_idx]

# ==============================================================================
# 5. Generación Gráfica (Fondo Blanco Plano y Paleta Contrastada)
# ==============================================================================
historial = modelo_xgb.evals_result()
rmse_tr_hist = historial['validation_0']['rmse']
rmse_te_hist = historial['validation_1']['rmse']

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5.8), facecolor='white')

# Panel 1: Curva de Convergencia del Error (RMSE)
ax1.set_facecolor('white')
rondas = range(1, len(rmse_tr_hist) + 1)
ax1.plot(rondas, rmse_tr_hist, color='#0072b2', linewidth=2, label='RMSE Entrenamiento')
ax1.plot(rondas, rmse_te_hist, color='#d55e00', linewidth=2.2, label='RMSE Prueba')
ax1.axhline(min(rmse_te_hist), color='#8a8a8a', linestyle=':', label=f'Mínimo Test: {min(rmse_te_hist):.1f}')
ax1.set_title("1. Convergencia del Error (RMSE)\na través de Rondas de Boosting", fontsize=11.5, fontweight='bold', pad=12)
ax1.set_xlabel("Número de Árboles (Iteraciones)", fontsize=10.5)
ax1.set_ylabel("Raíz del Error Cuadrático Medio (RMSE)", fontsize=10.5)
ax1.grid(True, linestyle=':', alpha=0.6)
ax1.legend(frameon=True, facecolor='white', loc='upper right', fontsize=9.5)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Panel 2: Valores Reales vs. Predichos
ax2.set_facecolor('white')
ax2.scatter(y_te, y_pred, color='#009e73', alpha=0.75, edgecolors='#222222', linewidth=0.5, s=55, label='Pacientes de Test (N=111)')
min_v = min(y_te.min(), y_pred.min()) - 10
max_v = max(y_te.max(), y_pred.max()) + 10
ax2.plot([min_v, max_v], [min_v, max_v], color='#d55e00', linestyle='--', linewidth=2, label='Predicción Perfecta (y = ŷ)')
ax2.set_xlim(min_v, max_v)
ax2.set_ylim(min_v, max_v)
ax2.set_title(f"2. Valores Reales vs. Predichos en Test\nR² = {r2_val:.3f} | RMSE = {rmse_val:.1f} | MAE = {mae_val:.1f}", fontsize=11.5, fontweight='bold', pad=12)
ax2.set_xlabel("Progresión Real de Diabetes (1 año)", fontsize=10.5)
ax2.set_ylabel("Progresión Predicha por XGBoost", fontsize=10.5)
ax2.grid(True, linestyle=':', alpha=0.6)
ax2.legend(frameon=True, facecolor='white', loc='upper left', fontsize=9.5)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Panel 3: Importancia de Atributos (Gain vs. Permutación)
ax3.set_facecolor('white')
y_pos = np.arange(len(vars_ordenadas))
h = 0.35
ax3.barh(y_pos - h/2, [importancias_gain[i] for i in top_idx], height=h, color='#0072b2', label='Ganancia (Gain en Train)')
ax3.barh(y_pos + h/2, [perm_res.importances_mean[i] for i in top_idx], height=h, color='#e69f00', label='Permutación (Caída R² en Test)')
ax3.set_yticks(y_pos)
ax3.set_yticklabels(vars_ordenadas, fontsize=9.5)
ax3.set_xlabel("Importancia Relativa", fontsize=10.5)
ax3.set_title("3. Importancia de Atributos:\nGanancia (Train) vs. Permutación (Test)", fontsize=11.5, fontweight='bold', pad=12)
ax3.grid(True, axis='x', linestyle=':', alpha=0.6)
ax3.legend(frameon=True, facecolor='white', loc='lower right', fontsize=9.5)
ax3.spines['top'].set_visible(False)
ax3.spines['right'].set_visible(False)

fig.suptitle("Análisis de Regresión con XGBoost Regressor (Dataset Real de Diabetes - Efron et al.)",
             fontsize=13.5, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

ModuleNotFoundError: No module named 'xgboost'